# Экспериментальная панель для курсовой: low-rank matrix completion

Эта версия заменяет черновой ноутбук, где эксперименты были собраны в разнобой и часть ячеек зависела от отсутствующего `synthetic_research_api.py`.

Здесь используется только реальный интерфейс из двух файлов проекта:

- `synthetic_api.py` — сборка сценариев, запуск методов, метрики, таблицы и графики;
- `matrix_completion_methods.py` — численные реализации методов.

Логика ноутбука:

1. фиксируем базовый синтетический сценарий;
2. задаем общий набор методов;
3. каждый эксперимент оформляем отдельной ячейкой;
4. для каждого эксперимента сохраняем raw-таблицу, агрегированную таблицу, сравнение с `Compact RGD + L2`, график RMSE и график времени;
5. в конце сохраняем общие таблицы по всем экспериментам.

Во всех основных sweep-экспериментах методы сравниваются с регуляризованным compact RGD. Это главный метод, вокруг которого строится интерпретация: он дает тот же риманов градиентный шаг, но хранит решение в compact USV-представлении и поэтому должен быть выгоднее по времени при похожем качестве восстановления.

In [ ]:
from pathlib import Path
import sys
import json
import shutil
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_DIR = Path.cwd().resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import synthetic_api as api

OUTPUT_DIR = PROJECT_DIR / 'research_outputs' / 'coursework_panel_v2'
FIG_DIR = OUTPUT_DIR / 'figures'
TABLE_DIR = OUTPUT_DIR / 'tables'
PKL_DIR = OUTPUT_DIR / 'dataframes_pickle'
for path in [OUTPUT_DIR, FIG_DIR, TABLE_DIR, PKL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

SEEDS = [41, 42, 43]
BASELINE_LABEL = 'Compact RGD + L2'
MANIFEST = []

print('project:', PROJECT_DIR)
print('outputs:', OUTPUT_DIR)

## 0. Общие настройки

Базовый сценарий небольшой, чтобы все sweep-и можно было запускать прямо из ноутбука. Размер 40×40 достаточен для проверки поведения методов, но не превращает ноутбук в тяжелый benchmark. Для финальных больших таблиц можно заменить `m=n=40` на больший размер и оставить те же ячейки.

Матрица строится как низкоранговая:

\[
X = U \operatorname{diag}(\sigma_1,\ldots,\sigma_r)V^\top,
\]

после чего часть элементов скрывается маской наблюдений. Метрики считаются на отложенной test-mask:

\[
\operatorname{RMSE}_{\Omega_{test}}
= \sqrt{\frac{1}{|\Omega_{test}|}\sum_{(i,j)\in \Omega_{test}}
(\widehat X_{ij}-X_{ij})^2}.
\]

In [ ]:
BASE_CONFIG = api.ScenarioConfig(
    matrix=api.MatrixConfig(
        m=40,
        n=40,
        rank=4,
        factor_distribution='gaussian',
        singular_value_profile='linear',
        singular_scale=1.0,
        coherence_mode='incoherent',
    ),
    structure=api.StructureConfig(mode='none'),
    missingness_field=api.MissingnessFieldConfig(mode='random'),
    mask_sampling=api.MaskSamplingConfig(observed_fraction=0.35, exact_fraction=True),
    split=api.SplitConfig(validation_fraction=0.15, test_fraction=0.15),
    noise=api.NoiseConfig(mode='gaussian', std=0.02),
)

STANDARD_METHODS = [
    api.make_method('soft_impute', label='Soft-Impute', max_iter=25, lambda_scale=0.15),
    api.make_method('als', label='ALS', max_iter=20, reg=1e-3, init='spectral'),
    api.make_method('rgd', label='RGD', max_iter=25, init='spectral'),
    api.make_method('rgd_l2', label='RGD + L2', max_iter=25, init='spectral', l2_reg=0.03),
    api.make_method('rgd_l2_selected', label='RGD + L2 selected', max_iter=20, init='spectral', l2_grid=[1e-3, 1e-2, 3e-2]),
    api.make_method('compact_rgd', label='Compact RGD', max_iter=25, init='spectral'),
    api.make_method('compact_rgd_l2', label='Compact RGD + L2', max_iter=25, init='spectral', l2_reg=0.03),
    api.make_method('compact_l2_selected', label='Compact RGD + L2 selected', max_iter=20, init='spectral', l2_grid=[1e-3, 1e-2, 3e-2]),
    api.make_method('factorized_gd', label='Factorized GD + L2', max_iter=35, init='spectral', l2_reg=0.03),
]

pd.DataFrame([
    {'method': m.name, 'label': m.label, **m.params}
    for m in STANDARD_METHODS
])

## 0.1. Вспомогательные функции

Они нужны только для оформления: запуск sweep-а, сохранение таблиц, сравнение с `Compact RGD + L2`, построение двух стандартных графиков.

In [ ]:
AGG_METRICS = (
    'test_rmse',
    'test_mae',
    'validation_rmse',
    'relative_fro_error',
    'runtime_sec',
    'iterations',
    'train_rmse_observed',
)


def save_df(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    MANIFEST.append(str(path.relative_to(PROJECT_DIR)))
    pkl_path = PKL_DIR / (path.stem + '.pkl')
    df.to_pickle(pkl_path)
    MANIFEST.append(str(pkl_path.relative_to(PROJECT_DIR)))
    return path


def compare_with_baseline(summary_df, x_col, baseline_label=BASELINE_LABEL):
    base = summary_df[summary_df['label'] == baseline_label][[
        x_col, 'avg_test_rmse', 'avg_runtime_sec', 'avg_relative_fro_error'
    ]].rename(columns={
        'avg_test_rmse': 'baseline_avg_test_rmse',
        'avg_runtime_sec': 'baseline_avg_runtime_sec',
        'avg_relative_fro_error': 'baseline_avg_relative_fro_error',
    })
    out = summary_df.merge(base, on=x_col, how='left')
    out['rmse_minus_baseline'] = out['avg_test_rmse'] - out['baseline_avg_test_rmse']
    out['runtime_ratio_to_baseline'] = out['avg_runtime_sec'] / out['baseline_avg_runtime_sec'].replace(0, np.nan)
    out['relative_fro_minus_baseline'] = out['avg_relative_fro_error'] - out['baseline_avg_relative_fro_error']
    return out


def plot_and_save(summary_rows, x_col, experiment_name, x_label=None):
    rmse_path = FIG_DIR / f'{experiment_name}_rmse.png'
    time_path = FIG_DIR / f'{experiment_name}_runtime.png'
    fro_path = FIG_DIR / f'{experiment_name}_relative_fro.png'

    api.plot_metric(
        summary_rows,
        x=x_col,
        y='avg_test_rmse',
        hue='label',
        title=f'{experiment_name}: test RMSE',
        xlabel=x_label or x_col,
        ylabel='avg test RMSE',
        save_path=rmse_path,
    )
    plt.close('all')
    api.plot_metric(
        summary_rows,
        x=x_col,
        y='avg_runtime_sec',
        hue='label',
        title=f'{experiment_name}: runtime',
        xlabel=x_label or x_col,
        ylabel='avg runtime, sec',
        save_path=time_path,
    )
    plt.close('all')
    api.plot_metric(
        summary_rows,
        x=x_col,
        y='avg_relative_fro_error',
        hue='label',
        title=f'{experiment_name}: relative Frobenius error',
        xlabel=x_label or x_col,
        ylabel='avg relative Frobenius error',
        save_path=fro_path,
    )
    plt.close('all')

    for path in [rmse_path, time_path, fro_path]:
        MANIFEST.append(str(path.relative_to(PROJECT_DIR)))


def run_one_factor_experiment(experiment_name, parameter_path, values, *, x_label=None, base_config=BASE_CONFIG, methods=STANDARD_METHODS, seeds=SEEDS):
    print(f'[{experiment_name}] {parameter_path} = {values}')
    runs = api.one_factor_sweep(
        base_config=base_config,
        parameter_path=parameter_path,
        values=values,
        methods=methods,
        seeds=seeds,
    )
    records = api.collect_records(runs)
    summary = api.aggregate_records(records, by=[parameter_path, 'label'], metrics=AGG_METRICS)

    records_df = pd.DataFrame(records)
    summary_df = pd.DataFrame(summary)
    comparison_df = compare_with_baseline(summary_df, parameter_path)

    save_df(records_df, TABLE_DIR / f'{experiment_name}_raw.csv')
    save_df(summary_df, TABLE_DIR / f'{experiment_name}_summary.csv')
    save_df(comparison_df, TABLE_DIR / f'{experiment_name}_vs_compact_l2.csv')

    plot_and_save(summary, parameter_path, experiment_name, x_label=x_label)

    print('raw rows:', len(records_df), 'summary rows:', len(summary_df))
    display(summary_df.head(20))
    display(comparison_df.sort_values([parameter_path, 'rmse_minus_baseline']).head(18))
    return {'runs': runs, 'records': records_df, 'summary': summary_df, 'comparison': comparison_df}


def run_custom_experiment(experiment_name, configs, methods, seeds, *, by, x_col, x_label=None, baseline_label=BASELINE_LABEL):
    print(f'[{experiment_name}] configs={len(configs)}, seeds={len(seeds)}, methods={len(methods)}')
    runs = api.run_many_experiments(configs, methods, seeds)
    records = api.collect_records(runs)
    summary = api.aggregate_records(records, by=by, metrics=AGG_METRICS)

    records_df = pd.DataFrame(records)
    summary_df = pd.DataFrame(summary)
    comparison_df = compare_with_baseline(summary_df, x_col, baseline_label=baseline_label)

    save_df(records_df, TABLE_DIR / f'{experiment_name}_raw.csv')
    save_df(summary_df, TABLE_DIR / f'{experiment_name}_summary.csv')
    save_df(comparison_df, TABLE_DIR / f'{experiment_name}_vs_compact_l2.csv')

    plot_and_save(summary, x_col, experiment_name, x_label=x_label)

    print('raw rows:', len(records_df), 'summary rows:', len(summary_df))
    display(summary_df.head(20))
    display(comparison_df.sort_values([x_col, 'rmse_minus_baseline']).head(18))
    return {'runs': runs, 'records': records_df, 'summary': summary_df, 'comparison': comparison_df}


def plot_singular_values(X, title, save_path=None):
    s = np.linalg.svd(X, compute_uv=False)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(range(1, len(s) + 1), s, marker='o')
    ax.set_title(title)
    ax.set_xlabel('index')
    ax.set_ylabel('singular value')
    ax.grid(alpha=0.3)
    fig.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=180)
        MANIFEST.append(str(Path(save_path).relative_to(PROJECT_DIR)))
    return fig, ax

## 1. Базовая проверка: один сценарий, все методы

Цель: убедиться, что пайплайн собирает матрицу, маску, запускает методы и сохраняет базовые результаты. Здесь же сохраняются картинки матрицы, маски и спектра.

In [ ]:
baseline_runs = api.run_many_experiments([BASE_CONFIG], STANDARD_METHODS, SEEDS)
baseline_records = pd.DataFrame(api.collect_records(baseline_runs))
baseline_summary = pd.DataFrame(api.aggregate_records(baseline_records.to_dict('records'), by=['label'], metrics=AGG_METRICS))
baseline_comparison = compare_with_baseline(baseline_summary, 'label')

save_df(baseline_records, TABLE_DIR / '01_baseline_raw.csv')
save_df(baseline_summary, TABLE_DIR / '01_baseline_summary.csv')
save_df(baseline_comparison, TABLE_DIR / '01_baseline_vs_compact_l2.csv')

scenario = api.assemble_scenario(BASE_CONFIG, seed=SEEDS[0])
api.plot_matrix(scenario.X_structured, title='Base structured matrix', save_path=FIG_DIR / '01_base_matrix.png')
MANIFEST.append(str((FIG_DIR / '01_base_matrix.png').relative_to(PROJECT_DIR)))
plt.close('all')
api.plot_mask(scenario.mask_split.observed_mask, title='Base observed mask', save_path=FIG_DIR / '01_base_mask.png')
MANIFEST.append(str((FIG_DIR / '01_base_mask.png').relative_to(PROJECT_DIR)))
plt.close('all')
plot_singular_values(scenario.X_structured, 'Base singular values', save_path=FIG_DIR / '01_base_singular_values.png')
plt.close('all')
api.plot_histories(baseline_runs[0].method_results, title='Base convergence histories', save_path=FIG_DIR / '01_base_histories.png')
MANIFEST.append(str((FIG_DIR / '01_base_histories.png').relative_to(PROJECT_DIR)))
plt.close('all')

display(baseline_summary.sort_values('avg_test_rmse').head(20))
display(baseline_comparison.sort_values('rmse_minus_baseline'))

## 2. Эксперимент: устойчивость к шуму

Меняем стандартное отклонение добавленного гауссовского шума в наблюдаемых элементах. Этот эксперимент показывает, насколько методы переобучаются под шум и помогает объяснить, зачем нужна L2-регуляризация.

In [ ]:
noise_experiment = run_one_factor_experiment(
    '02_noise_robustness',
    'noise.std',
    [0.0, 0.01, 0.02, 0.05, 0.10],
    x_label='noise std',
)

## 3. Эксперимент: зависимость от инициализации

Здесь меняется не сценарий, а вариант запуска методов: spectral initialization против random initialization. Для Soft-Impute отдельной инициализации нет, поэтому он оставлен как стабильная точка сравнения.

In [ ]:
INIT_METHODS = [
    api.make_method('soft_impute', label='Soft-Impute / no init', max_iter=25, lambda_scale=0.15),
]
for init in ['spectral', 'random']:
    INIT_METHODS.extend([
        api.make_method('als', label=f'ALS / {init}', max_iter=20, reg=1e-3, init=init),
        api.make_method('rgd', label=f'RGD / {init}', max_iter=25, init=init),
        api.make_method('rgd_l2', label=f'RGD + L2 / {init}', max_iter=25, init=init, l2_reg=0.03),
        api.make_method('compact_rgd', label=f'Compact RGD / {init}', max_iter=25, init=init),
        api.make_method('compact_rgd_l2', label=f'Compact RGD + L2 / {init}', max_iter=25, init=init, l2_reg=0.03),
        api.make_method('factorized_gd', label=f'Factorized GD + L2 / {init}', max_iter=35, init=init, l2_reg=0.03),
    ])

init_runs = api.run_many_experiments([BASE_CONFIG], INIT_METHODS, SEEDS)
init_records = pd.DataFrame(api.collect_records(init_runs))
init_summary = pd.DataFrame(api.aggregate_records(init_records.to_dict('records'), by=['label'], metrics=AGG_METRICS))
init_comparison = compare_with_baseline(init_summary, 'label', baseline_label='Compact RGD + L2 / spectral')

save_df(init_records, TABLE_DIR / '03_initialization_raw.csv')
save_df(init_summary, TABLE_DIR / '03_initialization_summary.csv')
save_df(init_comparison, TABLE_DIR / '03_initialization_vs_compact_l2_spectral.csv')

# Для categorical axis используем label как x и hue тоже label: график получается как ranked profile.
api.plot_metric(
    init_summary.sort_values('avg_test_rmse').to_dict('records'),
    x='label',
    y='avg_test_rmse',
    hue='label',
    title='03_initialization: test RMSE by initialization',
    xlabel='method / init',
    ylabel='avg test RMSE',
    save_path=FIG_DIR / '03_initialization_rmse.png',
)
MANIFEST.append(str((FIG_DIR / '03_initialization_rmse.png').relative_to(PROJECT_DIR)))
plt.close('all')
api.plot_metric(
    init_summary.sort_values('avg_runtime_sec').to_dict('records'),
    x='label',
    y='avg_runtime_sec',
    hue='label',
    title='03_initialization: runtime by initialization',
    xlabel='method / init',
    ylabel='avg runtime, sec',
    save_path=FIG_DIR / '03_initialization_runtime.png',
)
MANIFEST.append(str((FIG_DIR / '03_initialization_runtime.png').relative_to(PROJECT_DIR)))
plt.close('all')

print('raw rows:', len(init_records), 'summary rows:', len(init_summary))
display(init_summary.sort_values('avg_test_rmse').head(20))
display(init_comparison.sort_values('rmse_minus_baseline'))

## 4. Эксперимент: тип пропусков

Меняем механизм пропусков: случайные пропуски, блок, несколько кластеров, концентрация в строке/столбце, зависимость пропуска от величины элемента. Это важнее iid-маски, потому что реальные пропуски часто структурированы.

In [ ]:
missingness_experiment = run_one_factor_experiment(
    '04_missingness_type',
    'missingness_field.mode',
    ['random', 'block', 'clustered', 'row_focus', 'column_focus', 'value_based'],
    x_label='missingness type',
)

# Сохраняем примеры масок для каждого типа пропусков.
for mode in ['random', 'block', 'clustered', 'row_focus', 'column_focus', 'value_based']:
    cfg = api.apply_overrides(BASE_CONFIG, {'missingness_field.mode': mode})
    sc = api.assemble_scenario(cfg, seed=SEEDS[0])
    api.plot_mask(sc.mask_split.observed_mask, title=f'Observed mask: {mode}', save_path=FIG_DIR / f'04_mask_{mode}.png')
    MANIFEST.append(str((FIG_DIR / f'04_mask_{mode}.png').relative_to(PROJECT_DIR)))
    plt.close('all')

## 5. Эксперимент: процент наблюдаемых элементов

Меняем долю наблюдаемых элементов. Это прямой тест sample efficiency: насколько быстро методы начинают восстанавливать матрицу при росте информации.

In [ ]:
observed_fraction_experiment = run_one_factor_experiment(
    '05_observed_fraction',
    'mask_sampling.observed_fraction',
    [0.20, 0.28, 0.35, 0.45, 0.60, 0.75],
    x_label='observed fraction',
)

## 6. Эксперимент: распределение факторов матрицы

Меняем распределение, из которого генерируются факторы `U` и `V` до ортонормировки: gaussian, uniform, rademacher. Это проверяет, не является ли результат артефактом одного способа генерации данных.

In [ ]:
factor_distribution_experiment = run_one_factor_experiment(
    '06_factor_distribution',
    'matrix.factor_distribution',
    ['gaussian', 'uniform', 'rademacher'],
    x_label='factor distribution',
)

## 7. Эксперимент: распределение спектра

Меняем профиль сингулярных чисел: flat, linear, geometric. В квадратном случае это можно воспринимать как контроль распределения собственных значений низкорангового ядра. Геометрический спад обычно проще для восстановления, потому что эффективный ранг ниже.

In [ ]:
singular_profile_experiment = run_one_factor_experiment(
    '07_singular_value_profile',
    'matrix.singular_value_profile',
    ['flat', 'linear', 'geometric'],
    x_label='singular value profile',
)

## 8. Дополнительный эксперимент: когерентность сингулярных векторов

Low-rank completion хуже работает, если сингулярные векторы сконцентрированы в отдельных строках или столбцах. Поэтому отдельно проверяем `incoherent`, `coherent_rows`, `coherent_cols`, `coherent_both`.

In [ ]:
coherence_experiment = run_one_factor_experiment(
    '08_coherence_stress',
    'matrix.coherence_mode',
    ['incoherent', 'coherent_rows', 'coherent_cols', 'coherent_both'],
    x_label='coherence mode',
)

## 9. Дополнительный эксперимент: истинный ранг

Меняем истинный ранг матрицы. Методы получают тот же ранг, что и генератор сценария, поэтому здесь проверяется сложность восстановления, а не ошибка выбора ранга.

In [ ]:
rank_experiment = run_one_factor_experiment(
    '09_true_rank',
    'matrix.rank',
    [2, 4, 6, 8],
    x_label='true rank',
)

## 10. Общая сводка и сохранение пакета

Собираем единые таблицы по всем экспериментам. Они нужны, чтобы потом быстро вставлять результаты в TeX или строить новые графики без повторного запуска ноутбука.

In [ ]:
experiment_objects = {
    '01_baseline': {
        'records': baseline_records,
        'summary': baseline_summary,
        'comparison': baseline_comparison,
    },
    '02_noise_robustness': noise_experiment,
    '03_initialization': {
        'records': init_records,
        'summary': init_summary,
        'comparison': init_comparison,
    },
    '04_missingness_type': missingness_experiment,
    '05_observed_fraction': observed_fraction_experiment,
    '06_factor_distribution': factor_distribution_experiment,
    '07_singular_value_profile': singular_profile_experiment,
    '08_coherence_stress': coherence_experiment,
    '09_true_rank': rank_experiment,
}

all_records = []
all_summaries = []
all_comparisons = []
for name, obj in experiment_objects.items():
    rec = obj['records'].copy()
    summ = obj['summary'].copy()
    comp = obj['comparison'].copy()
    rec.insert(0, 'experiment', name)
    summ.insert(0, 'experiment', name)
    comp.insert(0, 'experiment', name)
    all_records.append(rec)
    all_summaries.append(summ)
    all_comparisons.append(comp)

all_records_df = pd.concat(all_records, ignore_index=True, sort=False)
all_summaries_df = pd.concat(all_summaries, ignore_index=True, sort=False)
all_comparisons_df = pd.concat(all_comparisons, ignore_index=True, sort=False)

save_df(all_records_df, TABLE_DIR / 'all_experiments_raw.csv')
save_df(all_summaries_df, TABLE_DIR / 'all_experiments_summary.csv')
save_df(all_comparisons_df, TABLE_DIR / 'all_experiments_vs_compact_l2.csv')

manifest_path = OUTPUT_DIR / 'manifest.json'
manifest_path.write_text(json.dumps(sorted(set(MANIFEST)), ensure_ascii=False, indent=2), encoding='utf-8')
MANIFEST.append(str(manifest_path.relative_to(PROJECT_DIR)))

readme = OUTPUT_DIR / 'README.md'
readme_text = (
    '# Coursework synthetic experiments\n\n'
    'Содержимое пакета:\n\n'
    '- `synthetic_research_panel_clean_executed.ipynb` — аккуратный выполненный ноутбук;\n'
    '- `tables/` — raw, summary и baseline comparison CSV по каждому эксперименту;\n'
    '- `dataframes_pickle/` — те же датафреймы в pickle-формате;\n'
    '- `figures/` — графики RMSE, runtime, relative Frobenius error и диагностические маски;\n'
    '- `synthetic_api.py`, `matrix_completion_methods.py` — код, на котором запускались эксперименты;\n'
    '- `synthetic_api_guide.txt`, `synthetic_api_documentation.txt` — короткий гайд и документация интерфейса.\n'
)
readme.write_text(readme_text, encoding='utf-8')
MANIFEST.append(str(readme.relative_to(PROJECT_DIR)))

display(all_summaries_df.head(30))
print('total raw rows:', len(all_records_df))
print('saved files:', len(set(MANIFEST)))